# ResolveAI — Customer Support Agent

An implementation design for a Streamlit customer-support agent using Retrieval-Augmented Generation (RAG), Pinecone, and agent-to-agent (A2A) specialist handoffs.

**Outcomes:** grounded answers with citations, safe specialist routing, and human escalation when automation cannot confidently resolve an issue.

## Architecture

```text
Customer → Streamlit conversation UI → Conversation orchestrator
                                      ├─ RAG retriever → Pinecone knowledge base
                                      ├─ A2A router → Billing | Orders | Technical agents
                                      └─ Ticket queue → Human support
```

1. The orchestrator determines whether a request needs retrieval, a specialist, or an escalation.
2. LangGraph runs the stateful workflow: triage → retrieve knowledge → optional A2A handoff → compose.
3. The A2A graph node sends only scoped, non-sensitive context to the relevant specialist endpoint.
4. Account-changing actions require identity verification and should remain human-approved.

## Setup

Run this from the `customer support agent` directory after installing the project requirements. Pinecone and OpenAI are optional: without them, the notebook uses bundled demo knowledge.

In [ ]:
# Step 1: Install project dependencies once, then restart the kernel if prompted.
# %pip install -r requirements.txt
# Step 2: Import Path to locate files and sys to expose project modules to Jupyter.
from pathlib import Path
import sys

# Step 3: Use the current folder when the notebook is opened inside the project.
PROJECT_DIR = Path.cwd()
# Step 4: Otherwise locate the project when Jupyter started from its parent folder.
if not (PROJECT_DIR / 'services').exists():
    PROJECT_DIR = PROJECT_DIR / 'customer support agent'
# Step 5: Make services/ importable without packaging the app first.
sys.path.insert(0, str(PROJECT_DIR))
# Step 6: Display the resolved project folder as a quick environment check.
PROJECT_DIR

## Module 1 — Retrieval

`search()` first tries Pinecone when its credentials are configured. Otherwise it uses lightweight local keyword retrieval, so the interaction design remains testable offline. In production, add approved help-centre documents with `title`, `content`, and `category` metadata.

In [ ]:
# Step 1: Import the retrieval service used by the Streamlit application.
from services.retrieval import search

# Step 2: Submit a representative customer question.
sources, retrieval_mode = search('How long does a refund take?')
# Step 3: Confirm whether Pinecone or local demo retrieval served the answer.
print('Mode:', retrieval_mode)
# Step 4: Inspect each returned article, category, score, and evidence text.
for source in sources:
    print(f"• {source['title']} ({source['category']}) — {source['score']:.2f}")
    print(' ', source['content'])

## Module 2 — LangGraph A2A orchestration

The compiled LangGraph routes state through `triage → retrieve_knowledge → (a2a_handoff | compose) → compose`. When `A2A_BILLING_URL`, `A2A_ORDERS_URL`, or `A2A_TECHNICAL_URL` is set, the `a2a_handoff` node POSTs a compact A2A-style message envelope to that service. Otherwise it simulates the handoff so the UI can be evaluated locally.

In [ ]:
# Step 1: Import the compiled LangGraph workflow and its convenient invoke wrapper.
from services.support_graph import resolve_support_request, support_graph

# Step 2: Define a billing question that should be routed to a specialist.
customer_message = 'My renewal charge failed and I need my invoice.'
# Step 3: Invoke LangGraph; it triages, retrieves, conditionally hands off, then composes.
result = resolve_support_request(customer_message, conversation=[])
# Step 4: Inspect the A2A handoff payload and connection/simulation status.
result['handoff']

## Module 3 — Response orchestration

The response always prioritizes retrieved support knowledge. The specialist result is a recommendation, not an unreviewed policy source. If retrieval has no useful evidence, create a human escalation rather than inventing an answer.

In [ ]:
# Step 1: The same compiled graph is invoked by Streamlit for every message.
result = resolve_support_request('Where can I see my order tracking?', conversation=[])
# Step 2: Print the final grounded response that the UI will display.
print(result['answer'])

## UI design checklist

The companion `app.py` implements this in Streamlit:

- **Customer conversation:** chat interaction, retrieved evidence cards, and A2A-handoff status.
- **Operations desk:** active escalation tickets, handoff history, and support metrics.
- **Architecture view:** the flow from user request to retrieval, specialists, and human escalation.
- **Guardrails:** do not expose secrets or send passwords/full payment details to agents; verify identity before account actions; retain audit logs for retrieval and handoffs.

## Run the UI

```bash
streamlit run app.py
```

To build the bundled sample Pinecone index after setting API keys in `.env`:

```bash
python ingest.py
```